In [8]:
import pandas as pd
import sqlite3
import datetime

In [9]:
df = pd.read_csv("g61_Elearning_Certificates (2).csv", sep=';')

df = df.dropna(how="all").dropna(axis=1, how="all")

df["issue_date"] = pd.to_datetime(df["issue_date"], dayfirst=True, format="%Y-%m-%d", errors="coerce").dt.date
df["online_date"] = pd.to_datetime(df["online_date"], dayfirst=True, format="%Y-%m-%d", errors="coerce").dt.date

df["certificate_fee"] = df["certificate_fee"].astype(str).str.replace(',', '.', regex=False).astype(float)

df["course_id"] = pd.to_numeric(df["course_id"].astype(str).str.replace(',', '.', regex=False).str.replace(r"\.0$", "", regex=True), errors='coerce').astype("Int64")
df["course_info"] = df["course_info"].str.replace(';', ',', regex=False)

In [11]:
df_platform = (
    df[["platform_id"]]
    .dropna()
    .drop_duplicates()
    .astype({"platform_id": "Int64"})
    .reset_index(drop=True)
)


df_certificate = (
    df[["certificate_id", "certificate_name", "certificate_type"]]
    .dropna(subset=["certificate_id"])
    .drop_duplicates(subset=["certificate_id"])
    .astype({"certificate_id": "Int64"})
    .reset_index(drop=True)
)


df_course = (
    df[["course_id", "course_name", "course_info", "platform_id", "online_date"]]
    .loc[lambda x: x["course_id"].isna() | ~x.duplicated(subset=["course_id"])]
    .astype({"platform_id": "Int64"})
    .reset_index(drop=True)
)


df_trans = (
    df[["platform_id", "certificate_id", "issue_date", "certificate_fee"]]
    .dropna(subset=["platform_id", "certificate_id"])
    .astype({"platform_id": "Int64", "certificate_id": "Int64"})
    .reset_index(drop=True)
)


nulos_count = df_course["course_id"].isna().sum()

print(f'Platform: {len(df_platform)} plataformas únicas.')
print(f'Certificate: {len(df_certificate)} certificados únicos.')
print(f'Course: {len(df_course)} cursos (Números únicos + {nulos_count} Nulos).')
print(f'Transaction: {len(df_trans)} transações.')

Platform: 500 plataformas únicas.
Certificate: 1000 certificados únicos.
Course: 6835 cursos (Números únicos + 6635 Nulos).
Transaction: 10652 transações.


In [12]:
con = sqlite3.connect("novadatabase5.db")

df_platform.to_sql("Platform", con, if_exists="replace", index=False)
df_certificate.to_sql("Certificate", con, if_exists="replace", index=False)
df_course.to_sql("Course", con, if_exists="replace", index=False)
df_trans.to_sql("Trans", con, if_exists="replace", index=False)

con.close()
